In [3]:
import pandas as pd
import numpy as np

In [4]:
df = pd.read_csv("../data/hotel_bookings.csv")

In [5]:
print(df.shape)

(119390, 32)


In [6]:
print(df.info())

<class 'pandas.DataFrame'>
RangeIndex: 119390 entries, 0 to 119389
Data columns (total 32 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   hotel                           119390 non-null  str    
 1   is_canceled                     119390 non-null  int64  
 2   lead_time                       119390 non-null  int64  
 3   arrival_date_year               119390 non-null  int64  
 4   arrival_date_month              119390 non-null  str    
 5   arrival_date_week_number        119390 non-null  int64  
 6   arrival_date_day_of_month       119390 non-null  int64  
 7   stays_in_weekend_nights         119390 non-null  int64  
 8   stays_in_week_nights            119390 non-null  int64  
 9   adults                          119390 non-null  int64  
 10  children                        119386 non-null  float64
 11  babies                          119390 non-null  int64  
 12  meal                       

In [7]:
print(df.isnull().sum().sort_values(ascending=False).head(6))

company                     112593
agent                        16340
country                        488
children                         4
arrival_date_month               0
arrival_date_week_number         0
dtype: int64


In [8]:
print(df.duplicated().sum())

31994


In [9]:
print(df[['lead_time', 'adr', 'adults', 'children', 'babies']].describe())

           lead_time            adr         adults       children  \
count  119390.000000  119390.000000  119390.000000  119386.000000   
mean      104.011416     101.831122       1.856403       0.103890   
std       106.863097      50.535790       0.579261       0.398561   
min         0.000000      -6.380000       0.000000       0.000000   
25%        18.000000      69.290000       2.000000       0.000000   
50%        69.000000      94.575000       2.000000       0.000000   
75%       160.000000     126.000000       2.000000       0.000000   
max       737.000000    5400.000000      55.000000      10.000000   

              babies  
count  119390.000000  
mean        0.007949  
std         0.097436  
min         0.000000  
25%         0.000000  
50%         0.000000  
75%         0.000000  
max        10.000000  


In [10]:
print(df['is_canceled'].value_counts(normalize=True))

is_canceled
0    0.629584
1    0.370416
Name: proportion, dtype: float64


In [11]:
df_clean = df.copy()
print("Number of rows before cleaning: ", len(df_clean))

Number of rows before cleaning:  119390


In [12]:
df_clean = df_clean.drop(columns=['company'])

df_clean['agent'] = df_clean['agent'].fillna(0).astype(int)
df_clean['children'] = df_clean['children'].fillna(0).astype(int)
df_clean['country'] = df_clean['country'].fillna('Unknown')

print(df_clean.isnull().sum().sum())

0


In [13]:
before = len(df_clean)

# 1) Bookings with no guests
no_guests = (df_clean['adults'] + df_clean['children'] + df_clean['babies']) == 0
df_clean = df_clean[~no_guests]

# 2) Negative or highly anomalous prices
df_clean = df_clean[(df_clean['adr'] >= 0) & (df_clean['adr'] < 1000)]

# 3) Bookings with no overnight stays
no_nights = (df_clean['stays_in_weekend_nights'] + df_clean['stays_in_week_nights']) == 0
df_clean = df_clean[~no_nights]

print("Deleted rows: ", before - len(df_clean))
print("Current number of rows: ", len(df_clean))

Deleted rows:  827
Current number of rows:  118563


In [14]:
print("Remaining repetitions: ", df_clean.duplicated().sum())

Remaining repetitions:  31933


In [15]:
# Total nights and guests
df_clean['total_nights'] = df_clean['stays_in_weekend_nights'] + df_clean['stays_in_week_nights']
df_clean['total_guests'] = df_clean['adults'] + df_clean['children'] + df_clean['babies']

# Revenue and Lost Revenueع
df_clean['revenue'] = df_clean['adr'] * df_clean['total_nights']
df_clean['lost_revenue'] = np.where(df_clean['is_canceled'] == 1, df_clean['revenue'], 0)

# Full access date
df_clean['arrival_date'] = pd.to_datetime(
    df_clean['arrival_date_year'].astype(str) + '-' +
    df_clean['arrival_date_month'] + '-' +
    df_clean['arrival_date_day_of_month'].astype(str),
    format='%Y-%B-%d'
)
df_clean['arrival_month_num'] = df_clean['arrival_date'].dt.month

# Pre-booking duration categories
df_clean['lead_time_group'] = pd.cut(
    df_clean['lead_time'],
    bins=[-1, 7, 30, 90, 180, 365, 800],
    labels=['0-7 days', '8-30 days', '31-90 days', '91-180 days', '181-365 days', '365+ days']
)

# Did the customer receive a room other than the one they booked?
df_clean['room_changed'] = (df_clean['reserved_room_type'] != df_clean['assigned_room_type']).astype(int)

df_clean[['total_nights', 'revenue', 'lost_revenue', 'arrival_date', 'lead_time_group', 'room_changed']].head()

,total_nights,revenue,lost_revenue,arrival_date,lead_time_group,room_changed
2,1,75.0,0.0,2015-07-01,0-7 days,1
3,1,75.0,0.0,2015-07-01,8-30 days,0
4,2,196.0,0.0,2015-07-01,8-30 days,0
5,2,196.0,0.0,2015-07-01,8-30 days,0
6,2,214.0,0.0,2015-07-01,0-7 days,0


In [16]:
print(df_clean.shape)
print(df_clean[['adr', 'total_nights', 'revenue', 'total_guests']].describe())

print("Cancellation rate", round(df_clean['is_canceled'].mean() * 100, 2), "%")
print("Total lost revenue:", round(df_clean['lost_revenue'].sum(), 0))

df_clean.to_csv("../data/hotel_bookings_clean.csv", index=False)

(118563, 39)
                 adr   total_nights        revenue   total_guests
count  118563.000000  118563.000000  118563.000000  118563.000000
mean      102.480047       3.444852     360.220416       1.972167
std        47.579382       2.534900     335.455674       0.719095
min         0.000000       1.000000       0.000000       1.000000
25%        70.000000       2.000000     150.000000       2.000000
50%        95.000000       3.000000     268.000000       2.000000
75%       126.000000       4.000000     448.010000       2.000000
max       510.000000      69.000000    7590.000000      55.000000
Cancellation rate 37.26 %
Total lost revenue: 16721837.0
